In [ ]:
import requests
import time
import pandas as pd
from datetime import datetime
rows = []

API_KEY = "your_api_key_here"  # Replace with your actual OpenWeatherMap API key

cities = [
    "Delhi", "Mumbai", "Chennai", "Kolkata", "Bangalore","Hyderabad", "Pune", "Ahmedabad", "Jaipur", "Lucknow","Himachal Pradesh","punjab"
]


for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city},IN&appid={API_KEY}&units=metric"
    response = requests.get(url)
    data = response.json()
    
    row = {
        "city": data["name"],
        "recorded_at": datetime.fromtimestamp(data["dt"]).strftime("%Y-%m-%d %H:%M:%S"),
        "temperature": data["main"]["temp"],
        "feels_like": data["main"]["feels_like"],
        "temp_min": data["main"]["temp_min"],
        "temp_max": data["main"]["temp_max"],
        "pressure": data["main"]["pressure"],
        "humidity": data["main"]["humidity"],
        "wind_speed": data["wind"]["speed"],
        "wind_deg": data["wind"]["deg"],
        "cloud_cover": data["clouds"]["all"],
        "weather_main": data["weather"][0]["main"],
        "visibility": data.get("visibility", None),
        "sunrise": datetime.fromtimestamp(data["sys"]["sunrise"]).strftime("%H:%M:%S"),
        "sunset": datetime.fromtimestamp(data["sys"]["sunset"]).strftime("%H:%M:%S"),
        "latitude": data["coord"]["lat"],
        "longitude": data["coord"]["lon"],
    }
    
    rows.append(row)
    time.sleep(1)  # rate limit

df = pd.DataFrame(rows)
print(df.head())

        city          recorded_at  temperature  feels_like  temp_min  \
0      Delhi  2026-10-04 21:42:51        28.36       29.75     28.36   
1     Mumbai  2026-10-04 21:48:36        29.17       31.49     29.17   
2    Chennai  2026-10-04 21:46:13        29.87       36.87     29.87   
3    Kolkata  2026-10-04 21:48:31        27.52       31.44     27.52   
4  Bengaluru  2026-10-04 21:41:37        26.19       26.19     26.19   

   temp_max  pressure  humidity  wind_speed  wind_deg  cloud_cover  \
0     28.36      1011        58        2.80       108            0   
1     29.17      1013        61        3.79       338            8   
2     29.87      1014        80        4.14       126            2   
3     27.52      1011        84        1.84       213           45   
4     26.19      1016        67        2.08       109          100   

  weather_main  visibility   sunrise    sunset  latitude  longitude  
0        Clear       10000  06:15:21  18:04:24   28.6667    77.2167  
1     

In [8]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   - -------------------------------------- 0.8/17.7 MB 6.7 MB/s eta 0:00:03
   --------------- ------------------------ 6.8/17.7 MB 22.1 MB/s eta 0:00:01
   ---------------------------- ----------- 12.6/17.7 MB 25.9 MB/s eta 0:00:01
   -------------------------------------- - 17.0/17.7 MB 24.0 MB/s eta 0:00:01
   ---------------------------------------- 17.7/17.7 MB 22.1 MB/s  0:00:00


In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    port="your_port_here",
    user="root",
    password="Your Password Here",
)
cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS weather_db")
cursor.execute("USE weather_db")

cursor.execute("""
CREATE TABLE IF NOT EXISTS weather_data (
    id INT AUTO_INCREMENT PRIMARY KEY,
    city VARCHAR(100),
    recorded_at DATETIME,
    temperature DECIMAL(5,2),
    feels_like DECIMAL(5,2),
    temp_min DECIMAL(5,2),
    temp_max DECIMAL(5,2),
    pressure INT,
    humidity INT,
    wind_speed DECIMAL(5,2),
    wind_deg INT,
    cloud_cover INT,
    weather_main VARCHAR(50),
    visibility INT,
    sunrise TIME,
    sunset TIME,
    latitude DECIMAL(8,4),
    longitude DECIMAL(8,4),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()
cursor.close()
conn.close()
print("Database and table ready.")

Database and table ready.


In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    port="your_port_here",
    user="root",
    password="Your Password Here",
    database="weather_db"
)
cursor = conn.cursor()

insert_query = """
INSERT INTO weather_data
(city, recorded_at, temperature, feels_like, temp_min, temp_max,
 pressure, humidity, wind_speed, wind_deg, cloud_cover,
 weather_main, visibility, sunrise, sunset, latitude, longitude)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

for _, row in df.iterrows():
    cursor.execute(insert_query, (
        row["city"], row["recorded_at"], row["temperature"], row["feels_like"],
        row["temp_min"], row["temp_max"], row["pressure"], row["humidity"],
        row["wind_speed"], row["wind_deg"], row["cloud_cover"],
        row["weather_main"], row["visibility"], row["sunrise"],
        row["sunset"], row["latitude"], row["longitude"]
    ))

conn.commit()
cursor.close()
conn.close()
print(f"{len(df)} rows inserted into MySQL.")

12 rows inserted into MySQL.
